# Spatio-Temporal dynamics of the intertidal seagrass meadow of Banc

d’Arguin, Mauritania, in a context of high atmospheric complexity.

Simon Oiry [ORCID profile for Simon Oiry](https://orcid.org/0000-0001-7161-5246) (Institut des Substances et Organismes de la Mer, ISOMer, Nantes Université, UR 2160, F-44000 Nantes, France, Consiglio Nazionale delle Ricerche, Istituto di Scienze Marine (CNR-ISMAR), 34149 Trieste, Italy)  
Pierre Gernez [ORCID profile for Pierre Gernez](https://orcid.org/0000-0003-2055-410X) (Institut des Substances et Organismes de la Mer, ISOMer, Nantes Université, UR 2160, F-44000 Nantes, France)  
Laurent Barillé [ORCID profile for Laurent Barillé](https://orcid.org/0000-0001-5138-2684) (Institut des Substances et Organismes de la Mer, ISOMer, Nantes Université, UR 2160, F-44000 Nantes, France)  
El-Hacen Mohamed El-Hacen [ORCID profile for El-Hacen Mohamed El-Hacen](https://orcid.org/0000-0002-0628-7335) (BirdEyes, Centre for Global Ecological Change at the Faculties of Science & Engineering and Campus Fryslân, Wirdumerdijk, Leeuwarden, The Netherlands, Parc National du Banc d’Arguin, Tavregh Zeina, Nouakchott, Mauritania)  
Jim de Fouw [ORCID profile for Jim de Fouw](https://orcid.org/0000-0002-0922-1331) (Department of Coastal systems, NIOZ Royal Netherlands Institute for Sea Research, Texel, Netherlands)  
October 14, 2026

To be Written

In [ ]:
library(reticulate)
options(reticulate.conda_binary = "C:/Users/Simon/miniconda3/Scripts/conda.exe")

use_condaenv("ArguinSatellite_2026", required = TRUE,
             conda = "C:/Users/Simon/miniconda3/Scripts/conda.exe")

In [ ]:
set.seed(20261014)

# Figure 1: accepted Frida source, inlined for single-source reproducibility.
suppressPackageStartupMessages({
  library(ggplot2)
  library(ggrepel)
  library(patchwork)
  library(sf)
})

iteration <- 3L
project <- normalizePath(getwd(), mustWork = TRUE)
out_dir <- file.path(project, "figures")
luna_dir <- file.path(project, "workinprogress", "SCI-24r2_luna")
gpkg <- file.path(luna_dir, "frida_figure1_geodata.gpkg")
tile_dir <- file.path(project, "workinprogress", "SCI-44_arthur", "esri_tile_cache")
dir.create(tile_dir, recursive = TRUE, showWarnings = FALSE)

intertidal <- st_read(gpkg, layer = "intertidal_mask", quiet = TRUE) |> st_transform(3857)
extent_poly <- st_read(gpkg, layer = "banc_arguin_extent", quiet = TRUE) |> st_transform(3857)
stations <- st_read(gpkg, layer = "validation_stations", quiet = TRUE) |> st_transform(3857)
locator_land <- st_read(gpkg, layer = "west_africa_land_110m", quiet = TRUE)
stations_ll <- st_transform(stations, 4326)
station_xy <- cbind(st_drop_geometry(stations), as.data.frame(st_coordinates(stations)))
station_xy$surface_short <- sub("Microphytobenthos mud", "MPB mud", station_xy$surface_class, fixed=TRUE)
station_groups <- data.frame(
  label=c("Nearshore AC01–02", "Offshore AC03–08"),
  X=c(mean(station_xy$X[station_xy$station_id %in% c("AC01","AC02")]),
      mean(station_xy$X[station_xy$station_id %in% sprintf("AC%02d",3:8)])),
  Y=c(mean(station_xy$Y[station_xy$station_id %in% c("AC01","AC02")]),
      mean(station_xy$Y[station_xy$station_id %in% sprintf("AC%02d",3:8)]))
)

station_colours <- c(AC01="#332288", AC02="#88CCEE", AC03="#44AA99", AC04="#117733",
                     AC05="#999933", AC06="#DDCC77", AC07="#CC6677", AC08="#AA4499")

lonlat_to_tile <- function(lon, lat, zoom) {
  n <- 2^zoom
  c(x = floor((lon + 180) / 360 * n),
    y = floor((1 - log(tan(lat*pi/180) + 1/cos(lat*pi/180))/pi) / 2 * n))
}

tile_bounds_3857 <- function(x, y, zoom) {
  world <- 20037508.342789244
  size <- 2 * world / 2^zoom
  c(xmin = -world + x*size, xmax = -world + (x+1)*size,
    ymin = world - (y+1)*size, ymax = world - y*size)
}

get_tile <- function(x, y, zoom, provider = "esri") {
  path <- file.path(tile_dir, sprintf("%s_z%02d_x%d_y%d.png", provider, zoom, x, y))
  if (!file.exists(path) || file.info(path)$size == 0) {
    url <- if (provider == "osm") {
      sprintf("https://tile.openstreetmap.org/%d/%d/%d.png", zoom, x, y)
    } else {
      sprintf("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/%d/%d/%d", zoom, y, x)
    }
    download.file(url, path, mode = "wb", quiet = TRUE)
  }
  signature <- readBin(path, "raw", n = 2)
  image <- if (identical(signature, as.raw(c(0xff, 0xd8)))) jpeg::readJPEG(path) else png::readPNG(path)
  list(image = image, bounds = tile_bounds_3857(x, y, zoom))
}

tile_layer <- function(plot, x, y, zoom, provider = "esri") {
  tile <- get_tile(x, y, zoom, provider)
  b <- tile$bounds
  plot + annotation_raster(tile$image, b[["xmin"]], b[["xmax"]], b[["ymin"]], b[["ymax"]])
}

tile_plot_for_bbox <- function(bbox, zoom, provider = "esri") {
  ll <- st_transform(st_as_sfc(bbox), 4326) |> st_bbox()
  nw <- lonlat_to_tile(ll[["xmin"]], ll[["ymax"]], zoom)
  se <- lonlat_to_tile(ll[["xmax"]], ll[["ymin"]], zoom)
  p <- ggplot()
  for (x in seq(nw[["x"]], se[["x"]])) for (y in seq(nw[["y"]], se[["y"]])) {
    p <- tile_layer(p, x, y, zoom, provider)
  }
  p
}

main_bbox <- st_bbox(extent_poly)
main_pad <- c(x = 3000, y = 3000)
main_bbox[c("xmin","xmax")] <- main_bbox[c("xmin","xmax")] + c(-1,1)*main_pad[["x"]]
main_bbox[c("ymin","ymax")] <- main_bbox[c("ymin","ymax")] + c(-1,1)*main_pad[["y"]]
scale_10km_merc <- 10000 / cos(mean(st_coordinates(stations_ll)[,"Y"]) * pi/180)
scale_xend <- main_bbox[["xmax"]] - 1800
scale_xstart <- scale_xend - scale_10km_merc

main <- tile_plot_for_bbox(main_bbox, 11) +
  geom_sf(data = intertidal, fill = scales::alpha("#2DC7C9", .24), colour = "#74E0DE", linewidth = .22) +
  geom_sf(data = stations, aes(fill = station_id), shape = 21, size = 2.8,
          stroke = .7, colour = "white") +
  geom_label_repel(data = station_groups, aes(X, Y, label = label),
                   min.segment.length = 0, seed = 24, box.padding = .28,
                   point.padding = .35, size = 2.1, fontface = "bold",
                   fill = scales::alpha("white", .86), label.size = 0,
                   colour = "#202020", segment.colour = "white") +
  scale_fill_manual(values = station_colours, guide = "none") +
  coord_sf(xlim = main_bbox[c("xmin","xmax")], ylim = main_bbox[c("ymin","ymax")],
           expand = FALSE, datum = NA) +
  annotate("label", x=main_bbox[["xmax"]]-900, y=main_bbox[["ymax"]]-900,
           label="B", hjust=1, vjust=1, colour="black", fontface="bold", size=3.0,
           fill=scales::alpha("white",.86), label.size=0, label.padding=unit(1.2,"mm")) +
  annotate("segment", x=scale_xstart, xend=scale_xend,
           y=main_bbox[["ymin"]]+1700, yend=main_bbox[["ymin"]]+1700,
           colour="#171717", linewidth=1.1) +
  annotate("text", x=(scale_xstart+scale_xend)/2, y=main_bbox[["ymin"]]+2400,
           label="10 km", colour="#171717", size=2.25, fontface="bold") +
  theme_void(base_size = 8.5) +
  theme(plot.margin = margin(0,0,0,0), panel.border = element_rect(colour="#333333", fill=NA, linewidth=.45))

loc_pt <- st_union(stations_ll) |> st_centroid() |> st_transform(3857)
loc_label_xy <- st_coordinates(st_transform(st_sfc(st_point(c(-14.7,18.9)), crs=4326), 3857))[1,]
locator_bbox <- st_bbox(st_transform(st_as_sfc(st_bbox(c(xmin=-26, ymin=3, xmax=18, ymax=33),
                                                        crs=st_crs(4326))), 3857))
locator <- tile_plot_for_bbox(locator_bbox, 4, provider="osm") +
  geom_sf(data = loc_pt, shape = 21, fill = "#D43D32", colour = "white", size = 2.6, stroke = .7) +
  annotate("text", x = loc_label_xy[[1]], y = loc_label_xy[[2]], label = "Banc d'Arguin",
           hjust = 0, size = 2.25, fontface="bold") +
  coord_sf(xlim = locator_bbox[c("xmin","xmax")], ylim = locator_bbox[c("ymin","ymax")],
           expand = FALSE, datum=NA) +
  annotate("label", x=locator_bbox[["xmin"]], y=locator_bbox[["ymin"]], label="A",
           hjust=0, vjust=0, colour="black",
           fill=scales::alpha("white",.86), label.size=0, size=2.7, fontface="bold",
           label.padding=unit(1.1,"mm")) +
  theme_void(base_size = 8) +
  theme(panel.border = element_rect(fill=NA,colour="#333333",linewidth=.4),
        plot.margin=margin(0,0,0,0))

make_zoom <- function(ids, panel_letter, cluster_label) {
  s <- stations[stations$station_id %in% ids,]
  s_xy <- cbind(st_drop_geometry(s),as.data.frame(st_coordinates(s)))
  bb <- st_bbox(s)
  half <- 900
  centre <- c(mean(bb[c("xmin","xmax")]), mean(bb[c("ymin","ymax")]))
  bb <- st_bbox(c(xmin=centre[[1]]-half, ymin=centre[[2]]-half,
                  xmax=centre[[1]]+half, ymax=centre[[2]]+half), crs=st_crs(3857))
  tile_plot_for_bbox(bb, 16) +
    geom_sf(data=intertidal,fill=scales::alpha("#2DC7C9",.18),colour="#62D4D2",linewidth=.18) +
    geom_sf(data=s,aes(fill=station_id),shape=21,colour="white",size=3.1,stroke=.75) +
    geom_text_repel(data=s_xy,aes(X,Y,label=station_id),colour="white",fontface="bold",size=2.05,
                    seed=24,box.padding=.2,point.padding=.3,min.segment.length=0,
                    segment.colour="white",segment.size=.25,max.overlaps=Inf) +
    scale_fill_manual(values=station_colours,guide="none") +
    coord_sf(xlim=bb[c("xmin","xmax")],ylim=bb[c("ymin","ymax")],expand=FALSE,datum=NA) +
    annotate("label", x=bb[["xmin"]]+120, y=bb[["ymax"]]-120, label=panel_letter,
             hjust=0, vjust=1, colour="black", fontface="bold", size=2.7,
             fill=scales::alpha("white",.86), label.size=0,
             label.padding=unit(1.1,"mm")) +
    annotate("label", x=bb[["xmax"]]-120, y=bb[["ymax"]]-120, label=cluster_label,
             hjust=1, vjust=1, colour="black", fontface="bold", size=2.35,
             fill=scales::alpha("white",.86), label.size=0,
             label.padding=unit(.9,"mm")) +
    theme_void(base_size=7.2) +
    theme(panel.border=element_rect(colour="#333333",fill=NA,linewidth=.4),
          plot.margin=margin(0,0,0,0))
}

zoom_ids <- list(c("AC01","AC02"), sprintf("AC%02d",3:8))
zoom_letters <- c("C", "D")
zoom_labels <- c("Nearshore stations", "Offshore stations")
zooms <- Map(make_zoom, zoom_ids, zoom_letters, zoom_labels)
zoom_grid <- wrap_plots(zooms, ncol=1) +
  plot_annotation(theme=theme(plot.margin=margin(0,0,0,0)))

left <- main + inset_element(locator, left=0, bottom=0, right=.43, top=.25,
                             align_to="panel", on_top=TRUE)
fig1 <- (left | zoom_grid) + plot_layout(widths=c(1.50,1))

ggsave(file.path(out_dir,"StudyArea_Fig001.png"),fig1,width=183,height=150,units="mm",dpi=600,
       device=ragg::agg_png,bg="white")
ggsave(file.path(out_dir,"StudyArea_Fig001.svg"),fig1,width=183,height=150,units="mm",
       device=svglite::svglite,bg="white")


# Figures 2--4: accepted Frida source, inlined for single-source reproducibility.
suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(patchwork)
  library(readr)
  library(readxl)
  library(stringr)
  library(tidyr)
})

iteration <- 3L
project <- normalizePath(getwd(), mustWork = TRUE)
out_dir <- file.path(project, "figures")
p <- function(...) file.path(project, ...)

stations <- sprintf("AC%02d", 1:8)
surfaces <- c(AC01 = "Dry sand", AC02 = "Dry sand", AC03 = "MPB mud",
              AC04 = "MPB mud", AC05 = "Algal mud", AC06 = "White sand",
              AC07 = "Seagrass", AC08 = "Seagrass")
station_colours <- c(AC01="#332288", AC02="#88CCEE", AC03="#44AA99", AC04="#117733",
                     AC05="#999933", AC06="#DDCC77", AC07="#CC6677", AC08="#AA4499")
surface_shapes <- c("Dry sand"=21, "MPB mud"=22, "Algal mud"=23,
                    "White sand"=24, "Seagrass"=25)
bands <- c("B1","B2","B3","B4","B5","B6","B7","B8","B8A")
band_wavelength <- c(B1=442.25, B2=492.34, B3=558.95, B4=664.56, B5=703.89,
                     B6=739.10, B7=779.73, B8=832.88, B8A=864.76)
band_name <- c(B1="Coastal aerosol", B2="Blue", B3="Green", B4="Red",
               B5="Vegetation red edge", B6="Vegetation red edge",
               B7="Vegetation red edge", B8="NIR", B8A="Narrow NIR")
band_display <- c(B1="Coastal", B2="Blue", B3="Green", B4="Red",
                  B5="Red edge", B6="Red edge", B7="Red edge",
                  B8="NIR", B8A="Narrow NIR")
band_labels <- paste0(bands, "  ", unname(band_display[bands]))
# Frida-approved band palette; the final three bands darken from red to black.
band_colours <- c(B1="#4C17E2", B2="#6593F4", B3="#A3D06E", B4="#DF6F20",
                  B5="#BA0A0A", B6="#7F0000", B7="#590000", B8="#2C0000", B8A="#000000")

base_size <- c(`1`=8.3, `2`=8.6, `3`=8.8)[as.character(iteration)]
theme_pub <- theme_classic(base_size = base_size) +
  theme(axis.text = element_text(colour="#262626", size=base_size-1),
        axis.title = element_text(size=base_size),
        plot.title = element_text(size=base_size+.4, face="bold", hjust=0),
        strip.background = element_rect(fill="#F2F2F2", colour=NA),
        strip.text = element_text(face="bold", size=base_size),
        legend.title = element_text(face="bold", size=base_size-.2),
        legend.text = element_text(size=base_size-1),
        legend.key.height = unit(3.7, "mm"),
        legend.key.width = unit(6.2, "mm"),
        panel.spacing = unit(3.5, "mm"),
        plot.margin = margin(4, 5, 4, 5))

# Figure 2: field spectra and SRF-convolved values. One shared compact station legend.
asd <- read.table(p("Data","ASD","202603_Reflectance_avg.txt"), header=TRUE, check.names=FALSE) |>
  select(wl, matches("^AC_")) |>
  pivot_longer(-wl, names_to="raw", values_to="reflectance") |>
  mutate(station=str_replace(str_to_upper(raw), "AC_", "AC"))
srf <- read_excel(p("Data","COPE-GSEG-EOPG-TN-15-0007 - Sentinel-2 Spectral Response Functions 2024 - 4.0.xlsx"),
                  sheet="Spectral Responses (S2B)") |>
  rename(wl=SR_WL) |>
  pivot_longer(-wl, names_to="raw", values_to="srf") |>
  mutate(band=str_remove(raw,"^S2B_SR_AV_")) |>
  filter(band %in% bands, is.finite(srf), srf>0)
conv <- asd |>
  inner_join(select(srf,wl,band,srf), by="wl", relationship="many-to-many") |>
  summarise(reflectance=weighted.mean(reflectance,srf), .by=c(station,band)) |>
  mutate(band=factor(band,bands))
widths <- summarise(srf, xmin=min(wl), xmax=max(wl), .by=band)
station_labels <- setNames(unname(surfaces[stations]), stations)
station_lines <- c(AC03="dashed", setNames(rep("solid",7), stations[stations != "AC03"]))
surface_colours <- c("Dry sand"="#332288","MPB mud"="#117733","Algal mud"="#999933",
                     "White sand"="#DDCC77","Seagrass"="#AA4499")
asd <- asd |> mutate(surface=factor(unname(surfaces[station]),levels=names(surface_colours)))
conv <- conv |> mutate(surface=factor(unname(surfaces[station]),levels=names(surface_colours)))

p2a <- ggplot(filter(asd, between(wl,400,900)), aes(wl,reflectance,colour=surface,linetype=station)) +
  geom_line(linewidth=.62) +
  scale_colour_manual(values=surface_colours,drop=FALSE) +
  scale_linetype_manual(values=station_lines,breaks=stations,labels=station_labels,guide="none") +
  labs(title="A  Field spectra",x="Wavelength (nm)",y="ASD reflectance",colour=NULL,linetype=NULL) +
  theme_pub + theme(legend.position="none",plot.margin=margin(4,8,4,5))
p2b <- ggplot(conv,aes(band,reflectance,colour=surface,linetype=station,group=station)) +
  geom_line(linewidth=.58) + geom_point(size=1.45) +
  scale_colour_manual(values=surface_colours,drop=FALSE) +
  scale_linetype_manual(values=station_lines,breaks=stations,labels=station_labels,guide="none") +
  labs(title="B  MSI band-equivalent spectra",x="Sentinel-2B MSI band",y="Sentinel-2 simulated reflectance",
       colour=NULL,linetype=NULL) + theme_pub + theme(plot.margin=margin(4,5,4,8))
legend2 <- cowplot::get_legend(
  p2b + theme(legend.position="bottom",legend.box.margin=margin(-2,0,0,0)) +
    guides(colour=guide_legend(nrow=1,byrow=TRUE,
                               override.aes=list(linewidth=.7,size=1.6))))
fig2 <- ((p2a + (p2b + theme(legend.position="none")) + plot_layout(widths=c(1.12,1))) /
           wrap_elements(full=legend2) + plot_layout(heights=c(1,.13)))

# Figures 3--4: all eight stations and all nine bands from the canonical matchup table.
d <- read_csv(p("figures","matchup_all_stations.csv"),show_col_types=FALSE) |>
  mutate(target=recode(target,"Microphytobenthos mud"="MPB mud"),
         band=factor(band,bands), correction=factor(correction,c("Sen2Cor","ACOLITE")),
         wavelength=unname(band_wavelength[as.character(band)]))

metric_labels <- d |>
  summarise(RMSD=sqrt(mean(residual^2)), Bias=mean(residual), .by=correction) |>
  mutate(label=sprintf("RMSD %.3f  |  Bias %+.3f",RMSD,Bias))

make_p3 <- function(correction_name, show_shape=FALSE, show_band=FALSE) ggplot(filter(d,correction==correction_name),aes(asd_reflectance,satellite_reflectance)) +
  geom_abline(slope=1,intercept=0,linetype="22",colour="#777777",linewidth=.45) +
  geom_point(aes(shape=target,fill=band),size=if(iteration==1) 2.15 else 2.35,
             stroke=.45,colour="#202020",alpha=.9) +
  geom_label(data=filter(metric_labels, correction == correction_name),aes(x=.018,y=.685,label=label),inherit.aes=FALSE,
             hjust=0,vjust=1,size=2.45,label.size=0,fill=scales::alpha("white",.86)) +
  labs(title=correction_name) +
  scale_shape_manual(values=surface_shapes,drop=FALSE) +
  scale_fill_manual(values=band_colours,breaks=bands,labels=band_labels,drop=FALSE) +
  scale_x_continuous(breaks=seq(0,.6,.1),expand=expansion(mult=0)) +
  scale_y_continuous(breaks=seq(0,.7,.1),expand=expansion(mult=0)) +
  coord_equal(xlim=c(0,.6),ylim=c(0,.7),clip="off") +
  labs(x="ASD field reflectance",y="Satellite surface reflectance",
       shape="Surface class",fill="Band wavelength") + theme_pub +
  theme(legend.position=c(.975,.025),legend.justification=c(1,0),
        legend.background=element_rect(fill=scales::alpha("white",.9),colour="#BBBBBB",linewidth=.25),
        legend.margin=margin(1,1,1,1),legend.spacing.y=unit(.2,"mm"),
        legend.title=element_text(size=6.0,face="bold"),legend.text=element_text(size=5.2),
        legend.key.height=unit(2.35,"mm"),legend.key.width=unit(3.0,"mm")) +
  guides(shape=if(show_shape) guide_legend(title="Surface class",ncol=1,override.aes=list(fill="white",size=1.5)) else "none",
         fill=if(show_band) guide_legend(title="Band",ncol=2,byrow=TRUE,override.aes=list(shape=21,size=1.5)) else "none")

fig3 <- make_p3("Sen2Cor",show_shape=TRUE) + make_p3("ACOLITE",show_band=TRUE) +
  plot_layout(guides="keep") & theme(plot.margin=margin(4,7,4,7))

fig4 <- ggplot(d,aes(band,residual,colour=band)) +
  geom_hline(yintercept=0,linetype="22",colour="#666666",linewidth=.45) +
  geom_point(position=position_jitter(width=.13,height=0,seed=24),size=1.35,alpha=.65) +
  geom_boxplot(width=.68,outlier.shape=NA,fill=NA,linewidth=.55) +
  facet_grid(correction~.) +
  scale_colour_manual(values=band_colours,breaks=bands,labels=band_labels,drop=FALSE) +
  scale_y_continuous(breaks=seq(-.15,.15,.05),labels=function(x)sprintf("%.2f",x),
                     expand=expansion(mult=c(.06,.08))) +
  labs(x="Sentinel-2B MSI band",y="Residual (satellite − ASD reflectance)",colour="Band wavelength") +
  theme_pub + theme(legend.position="bottom",legend.box.margin=margin(-2,0,0,0),
                    legend.key.width=unit(4,"mm"),legend.text=element_text(size=6.6),
                    legend.title=element_text(size=7,face="bold"),
                    panel.spacing.y=unit(0,"mm"),
                    panel.border=element_rect(colour="#C9C9C9",fill=NA,linewidth=.3)) +
  guides(colour=guide_legend(nrow=2,byrow=TRUE,override.aes=list(linewidth=.35,size=1.1,alpha=1)))

save_figure <- function(plot, final_stem, width_mm, height_mm) {
  ggsave(file.path(out_dir,paste0(final_stem,".png")),plot,width=width_mm,height=height_mm,
         units="mm",dpi=600,device=ragg::agg_png,bg="white")
  ggsave(file.path(out_dir,paste0(final_stem,".svg")),plot,width=width_mm,height=height_mm,
         units="mm",device=svglite::svglite,bg="white")
}

save_figure(fig2,"FieldSpectra_Fig002",183,if(iteration==1) 105 else 118)
save_figure(fig3,"MatchupByBandAndSurface_Fig003",183,if(iteration==1) 115 else if(iteration==2) 140 else 120)
save_figure(fig4,"ResidualBoxplotsByBand_Fig004",183,if(iteration==1) 118 else 132)

# 1. Alternative Title propositions :

- Mapping intertidal seagrass dynamics under Saharan dust: atmospheric correction and Sentinel-2 time-series analysis in Banc d’Arguin
- Spatiotemporal dynamics of intertidal seagrass in Banc d’Arguin, Mauritania from Sentinel-2 imagery: overcoming atmospheric-correction challenges under Saharan dust

# 2. Introduction

Seagrasses are marine angiosperms that form highly productive meadows across shallow coastal waters ([Orth et al., 2006](#ref-Orth2006Crisis)). By creating three-dimensional habitat, attenuating waves, stabilising sediments and recycling nutrients, these foundation species support diverse assemblages of invertebrates, fishes, birds and marine megafauna ([Waycott et al., 2009](#ref-Waycott2009Loss)). Their ecological functions translate into services that include nursery habitat and fisheries support, shoreline protection, water-quality regulation and long-term organic-carbon storage ([Barbier et al., 2011](#ref-Barbier2011Value); [Fourqurean et al., 2012](#ref-Fourqurean2012Carbon); [Nordlund et al., 2016](#ref-Nordlund2016SeagrassServices); [Unsworth et al., 2019](#ref-Unsworth2019Challenges)). Yet seagrass meadows are exposed to multiple, often interacting pressures, including coastal development, eutrophication, altered sediment regimes, destructive fishing and anchoring, and climate-driven warming and extreme events ([Orth et al., 2006](#ref-Orth2006Crisis); [Waycott et al., 2009](#ref-Waycott2009Loss)). A synthesis of 547 sites estimated a net loss of 19.1% of the surveyed meadow area since 1884, while also revealing strong regional heterogeneity and recent recovery at some locations ([Dunic et al., 2021](#ref-Dunic2021Declines)). This variability makes sustained, spatially explicit observation essential: global trends cannot substitute for local evidence, and recovery cannot be distinguished from continued degradation without repeatable measurements ([Duffy et al., 2019](#ref-Duffy2019Observing)). The Parc National du Banc d’Arguin (PNBA), Mauritania, is an exceptional case. Located at the interface between the Sahara and the eastern boundary upwelling system of the North Atlantic, this approximately 12,000-km² protected seascape contains one of the largest remaining seagrass systems in Africa ([Pottier et al., 2021](#ref-Pottier2021Mapping); [Wolff et al., 1993](#ref-Wolff1993Banc)). A recent park-wide habitat map estimated about 772 km² of seagrass, comprising approximately 353 km² of intertidal meadow dominated by *Zostera noltei* and 419 km² of predominantly subtidal meadow dominated by *Cymodocea nodosa* ([Pottier et al., 2021](#ref-Pottier2021Mapping)). These meadows sustain the benthic food web, provide habitat and nursery grounds for fishes and invertebrates, and underpin a site used by more than two million migratory waterbirds along the East Atlantic Flyway ([El-Hacen et al., 2020](#ref-ElHacen2020LongTerm); [<span class="nocase">van der Heide et al.</span>, 2012](#ref-vanderHeide2012Symbiosis)). Their extent does not imply ecological stability: between spatially comprehensive surveys conducted in 1986 and 2014, seagrass cover increased approximately twofold while macrozoobenthic community composition and estimated secondary production changed markedly ([El-Hacen et al., 2020](#ref-ElHacen2020LongTerm)). The Banc d’Arguin meadow is therefore both a globally important conservation asset and a dynamic ecosystem whose trajectories require reliable observation ([<span class="nocase">de Fouw et al.</span>, 2016](#ref-deFouw2016Drought); [El-Hacen et al., 2020](#ref-ElHacen2020LongTerm)).

The environmental setting of Banc d’Arguin is governed by the juxtaposition of a hyper-arid desert margin, broad shallow tidal flats and the wind-driven Mauritanian upwelling system ([Ould Dedah, 1993](#ref-ould1993wind); [Wolff et al., 1993](#ref-Wolff1993Banc)). Freshwater input is negligible, precipitation is generally below 100 mm yr⁻¹, and persistent northerly winds help control waves, currents, sediment redistribution and exchanges between the bank and adjacent shelf ([Ould Dedah, 1993](#ref-ould1993wind); [Wolff et al., 1993](#ref-Wolff1993Banc)). Wind forcing is strongly seasonal: mean speeds reported for the intertidal system rise to about 9 m s⁻¹ in May–June and decline to about 6 m s⁻¹ in November–December ([Ould Dedah, 1993](#ref-ould1993wind)). This annual cycle is embedded within the latitudinal migration of the Intertropical Convergence Zone, which reorganises both the origin and vertical pathway of mineral-dust transport over northwestern Africa ([Knippertz and Todd, 2012](#ref-Knippertz2012Dust)). During winter, dust reaching coastal Mauritania is transported mainly in low-level north-easterly trade winds from Western Saharan and regional sources; during summer, transport increasingly occurs within the elevated Saharan Air Layer and draws on sources in Mauritania, Mali and farther east ([Friese et al., 2017](#ref-Friese2017Provenance); [Senghor et al., 2017](#ref-Senghor2017SeasonalCycle)). Superimposed on this seasonality are discrete dust outbreaks, including events that deliver anomalously coarse particles offshore ([Friese et al., 2016](#ref-Friese2016ParticleSize)). This aerosol regime creates a particular obstacle to optical monitoring of Banc d’Arguin, where low-tide habitat mapping depends on preserving the spectral consistency of exposed intertidal surfaces across acquisition dates ([Song et al., 2001](#ref-Song2001Atmospheric)). Each usable acquisition captures a highly heterogeneous coastal mosaic comprising dense and sparse *Zostera noltei*, wet and dry sediment, tidal pools and adjacent bright desert surfaces ([Pottier et al., 2021](#ref-Pottier2021Mapping)). Above this mosaic, mineral-dust abundance, particle-size distribution, source region and absorption properties vary seasonally and among individual outbreaks ([Friese et al., 2016](#ref-Friese2016ParticleSize); [Kaufman et al., 2002](#ref-Kaufman2002Aerosols); [Knippertz and Todd, 2012](#ref-Knippertz2012Dust)). These changes can alter the apparent brightness and spectral shape of otherwise unchanged intertidal targets, producing both spatial gradients within a scene and radiometric offsets among acquisition dates ([Doxani et al., 2023](#ref-Doxani2023ACIX); [Song et al., 2001](#ref-Song2001Atmospheric)). The difficulty is compounded by the way Sen2Cor, the atmospheric correction algorithm routinely applied on Sentinel-2 images by the European Space Agency, estimates aerosol optical thickness ([Main-Knorn et al., 2017](#ref-MainKnorn2017Sen2Cor)). The processor preferentially retrieves aerosol information over reference targets with known spectral behaviour, particularly dense dark vegetation, and uses fallback information or default values where suitable reference areas are unavailable ([European Space Agency, 2021](#ref-ESA2021Sen2CorATBD); [Main-Knorn et al., 2017](#ref-MainKnorn2017Sen2Cor)). In the hyper-arid Banc d’Arguin landscape, dominated by bright desert surfaces and an intertidal zone that alternates among vegetation, moist sediment, bare sediment and shallow pools, the availability and representativeness of such reference targets cannot be assumed ([Pottier et al., 2021](#ref-Pottier2021Mapping); [Wolff et al., 1993](#ref-Wolff1993Banc)). An aerosol model adequate for one acquisition may consequently leave a different residual bias in another ([Doxani et al., 2023](#ref-Doxani2023ACIX)). Thick, absorbing Saharan dust has already been shown to defeat conventional atmospheric correction off western Africa ([Moulin et al., 2001](#ref-Moulin2001Dust)), while comparative evaluations demonstrate that processor performance is strongly environment-dependent ([Bui et al., 2022](#ref-Bui2022Evaluation); [Doxani et al., 2023](#ref-Doxani2023ACIX); [Warren et al., 2019](#ref-Warren2019Assessment)). For this study, the central requirement is therefore not merely to retrieve plausible reflectance for a single date, but to preserve the field-measured spectral relationships among exposed intertidal targets across acquisitions made under contrasting dust conditions ([Doxani et al., 2023](#ref-Doxani2023ACIX); [Song et al., 2001](#ref-Song2001Atmospheric)).

Mapping exposed intertidal *Zostera noltei* requires stable spectral relationships among seagrass canopies, wet and dry sediment, macroalgae and residual tidal pools. Discrimination is particularly challenging where seagrasses and co-occurring green macroalgae, which share major photosynthetic pigments, produce partly overlapping multispectral signatures ([Davies et al., 2023](#ref-Davies2023Spectral)). Nevertheless, spectral-library experiments have shown that machine-learning classifiers can distinguish soft-bottom intertidal vegetation at Sentinel-2 spectral resolution, with little improvement obtained by increasing spectral resolution beyond approximately ten bands ([Davies et al., 2023](#ref-Davies2023Spectral)). Building on this foundation, Davies et al. developed the Intertidal Classification of Europe: Categorising Reflectance of Emerged Areas of Marine vegetation with Sentinel-2 (ICE CREAMS v1.0), a multiclass neural network trained with extensive drone-derived labels to identify seagrass and other exposed intertidal habitats ([Davies et al., 2024a](#ref-Davies2024Phenology)). Independent validation across approximately 12,000 Sentinel-2 pixels yielded an overall accuracy of 0.82 for seagrass presence or absence, and subsequent applications resolved both seasonal phenology across 12 North-East Atlantic meadows and contrasting interannual trajectories at six European sites ([Davies et al., 2024a](#ref-Davies2024Phenology), [2024b](#ref-Davies2024Trajectories)). These results establish ICE CREAMS as a scalable framework for intertidal seagrass monitoring, but transferring the classifier among dates and regions presupposes that the spectral relationships encoded by the model are preserved in the input imagery ([Davies et al., 2024a](#ref-Davies2024Phenology), [2024b](#ref-Davies2024Trajectories); [Song et al., 2001](#ref-Song2001Atmospheric)). Under the variable aerosol loads of Banc d’Arguin, incomplete correction can alter both the magnitude and shape of target spectra ([Doxani et al., 2023](#ref-Doxani2023ACIX); [Song et al., 2001](#ref-Song2001Atmospheric)). A classifier trained on one radiometric condition may then encounter another: unchanged seagrass can migrate in feature space, while atmospheric variability can be misinterpreted as habitat gain, loss or change in cover ([Song et al., 2001](#ref-Song2001Atmospheric)). This problem is particularly consequential for time-series analysis, for which consistency among dates is at least as important as the accuracy of any single scene ([Lyons et al., 2013](#ref-Lyons2013TimeSeries); [Roelfsema et al., 2014](#ref-Roelfsema2014Multitemporal)). Sentinel-2 has already demonstrated the feasibility of mapping the PNBA’s coastal habitats, with a single-date supervised classification attaining more than 80% overall accuracy ([Pottier et al., 2021](#ref-Pottier2021Mapping)). However, a static inventory cannot establish the timing, persistence or spatial pattern of ecological change ([Davies et al., 2024b](#ref-Davies2024Trajectories); [Lyons et al., 2013](#ref-Lyons2013TimeSeries)). Conversely, the pronounced differences observed between the 1986 and 2014 benthic surveys demonstrate the value of long-term comparison but do not resolve the intervening trajectory ([El-Hacen et al., 2020](#ref-ElHacen2020LongTerm)). A radiometrically consistent satellite record could bridge these scales by tracking expansion, contraction and fragmentation at frequencies unattainable through field surveys alone ([Davies et al., 2024a](#ref-Davies2024Phenology), [2024b](#ref-Davies2024Trajectories)). Without such a record, managers cannot confidently separate ecological change from retrieval artefacts, establish contemporary baselines, identify persistent hotspots of loss, evaluate conservation measures or target field surveillance ([Duffy et al., 2019](#ref-Duffy2019Observing); [Unsworth et al., 2019](#ref-Unsworth2019Challenges)). More broadly, incomplete information on meadow status and condition remains a major barrier to seagrass conservation, and coordinated observing frameworks explicitly call for the integration of field measurements with repeated satellite mapping ([Duffy et al., 2019](#ref-Duffy2019Observing); [Unsworth et al., 2019](#ref-Unsworth2019Challenges)). Resolving atmospheric-correction uncertainty is thus a prerequisite not only for classification accuracy, but also for defensible conservation assessments and policy decisions in the PNBA ([Doxani et al., 2023](#ref-Doxani2023ACIX); [Song et al., 2001](#ref-Song2001Atmospheric)).

This study addresses that bottleneck by establishing a locally validated workflow for mapping the intertidal seagrass meadows of Banc d’Arguin through time. First, we use field hyperspectral measurements acquired over representative seagrass and non-seagrass targets to evaluate Sentinel-2 reflectance produced by Sen2Cor and two alternative atmospheric-correction approaches. The field spectra are matched to the Sentinel-2 spectral response, enabling us to assess both spectral fidelity and temporal consistency and to identify the correction best suited to this dust-influenced coastal environment. Second, we adapt the ICE CREAMS classification framework to the selected reflectance product, so that the habitat model operates on radiometrically coherent inputs rather than on correction-dependent spectral artefacts. Third, we apply the resulting workflow to the Sentinel-2 archive to reconstruct a time series of seagrass distribution and quantify the spatial and temporal dynamics of the intertidal meadow. The following sections describe the field campaign, satellite processing and atmospheric-correction comparison, the adapted classification workflow, and the procedures used to validate maps and derive change metrics.

# 3. Materials and Methods

## 3.1 Study area and field reflectance measurements

The study was conducted in the Parc National du Banc d’Arguin (PNBA), north-western Mauritania, an extensive system of islands, shallow channels and intertidal flats at the interface between the Sahara and the eastern Atlantic ([Pottier et al., 2021](#ref-Pottier2021Mapping); [Wolff et al., 1993](#ref-Wolff1993Banc)). Fieldwork took place on 20–30 March 2026. Surface reflectance was measured with a portable ASD field spectroradiometer over 400–900 nm. On 27 March 2026, coincident with a Sentinel-2B overpass, eight stations (AC01–AC08) were sampled at two sites about 22 km apart (<a href="#fig-study-area" class="quarto-xref">Figure 1</a>): two on dry coastal sand (AC01–AC02), two on microphytobenthos (MPB) mud (AC03–AC04), one on algal mud (AC05), one on white sand (AC06) and two on *Zostera noltei* seagrass (AC07–AC08). At each station, spectra were recorded within an area of about one 10 m Sentinel-2 pixel (five sets of 20 spectra), and the average spectrum of the station was used as the field reference.

In [ ]:
knitr::include_graphics("figures/StudyArea_Fig001.png")

Each station spectrum was convolved to the bands of the Sentinel-2B Multispectral Instrument (MSI) that lie within the ASD range (B1–B8A; nine bands) using the official spectral response functions $S_b(\lambda)$ (ESA technical note COPE-GSEG-EOPG-TN-15-0007, version 4.0):

$$
R_b = \frac{\sum_{\lambda} R(\lambda)\,S_b(\lambda)}{\sum_{\lambda} S_b(\lambda)},
$$

where $R(\lambda)$ is the field reflectance at 1 nm and the sums run over the wavelengths covered by the ASD (<a href="#fig-field-spectra" class="quarto-xref">Figure 2</a>). Weighting the response functions by the extraterrestrial solar spectrum ([Wehrli, 1985](#ref-Wehrli1985Extraterrestrial)) changed the root-mean-square deviation (RMSD) of either processor by less than 0.0001 and is not used further.

In [ ]:
knitr::include_graphics("figures/FieldSpectra_Fig002.png")

## 3.2 Satellite reflectance products

Two surface-reflectance products were derived for the Sentinel-2B acquisition of 27 March 2026 (sensed at 11:45:53 UTC; tile 28QCH; processing baseline 05.12; cloud cover 0.015% of the tile). The first was the operational Level-2A product generated with Sen2Cor ([European Space Agency, 2021](#ref-ESA2021Sen2CorATBD); [Main-Knorn et al., 2017](#ref-MainKnorn2017Sen2Cor)), in which digital numbers were converted to bottom-of-atmosphere reflectance as $(\mathrm{DN} - 1000)/10{,}000$. Bands were used on their native grids (10 m: B2, B3, B4, B8; 20 m: B5, B6, B7, B8A; 60 m: B1). The second was obtained by processing the corresponding Level-1C product with ACOLITE using dark spectrum fitting ([Vanhellemont, 2019](#ref-Vanhellemont2019DSF); [Vanhellemont and Ruddick, 2018](#ref-Vanhellemont2018Metre)). Aerosol optical thickness was estimated in tiles of 600 × 600 pixels with minimum-DRMSD selection of the aerosol model and a three-pixel smoothing kernel. Surface reflectance (`rhos`) was produced at 10 m resolution, without adjacency or glint correction. ACOLITE bands were assigned to MSI bands according to the equivalent Sentinel-2B central wavelengths (maximum difference 0.34 nm).

## 3.3 Match-up of field and satellite reflectance

Station coordinates were projected to the coordinate system of each raster, and satellite reflectance was extracted from the pixel containing each station. The match-up comprised 72 paired values per processor (eight stations × nine bands). With $F_i$ the convolved field reflectance, $S_i$ the satellite reflectance and $d_i = S_i - F_i$ the residual, agreement was quantified by the RMSD and the mean bias,

$$
\mathrm{RMSD}=\sqrt{\tfrac{1}{n}\sum_i d_i^2},\qquad
\mathrm{bias}=\tfrac{1}{n}\sum_i d_i ,
$$

computed over all paired values, by band, by station and by surface class. Spectral-shape agreement was measured for each station by the spectral angle between the nine-band satellite and field vectors. The difference between processors was expressed as $\Delta\mathrm{RMSD} = \mathrm{RMSD}_{\mathrm{Sen2Cor}} - \mathrm{RMSD}_{\mathrm{ACOLITE}}$, so that positive values indicate a lower RMSD for ACOLITE.

Because the nine bands of a station are not independent, uncertainty was estimated by a station-cluster percentile bootstrap ([Field and Welsh, 2007](#ref-Field2007Bootstrapping)): stations were resampled with replacement, all bands of a sampled station were kept together, and RMSD, bias and $\Delta$RMSD were computed on the same draws for both processors (10,000 draws). The 2.5th and 97.5th percentiles are reported as 95% intervals. The influence of individual stations was assessed by recomputing $\Delta$RMSD after omitting each station in turn.

Three analytical choices were varied to test the robustness of the comparison: (i) exclusion of AC03, whose spectrum differs markedly between field and satellite products (see Results), giving a seven-station set (63 paired values per processor); (ii) extraction of both products on a common 10 m grid, using nearest-neighbour and bilinear sampling; and (iii) replacement of the single pixel by the mean of the 3 × 3 pixels (30 m) centred on each station.

## 3.4 Pixel-wise comparison of Sen2Cor and ACOLITE reflectance

To describe how the two processors differ across the landscape, independently of the field stations, the Sen2Cor and ACOLITE reflectance of the 27 March 2026 scene were compared pixel by pixel for the eleven bands common to both products (B1–B8, B8A, B11 and B12). B9 was not available in the ACOLITE export and B10 is not a surface-reflectance band. Comparisons were made on the 10 m ACOLITE grid of the study area (3,763 × 1,974 pixels, WGS 84 / UTM zone 28N). Sen2Cor reflectance was resampled to this grid by bilinear interpolation from its native 10, 20 or 60 m resolution, and the Sen2Cor scene classification layer (SCL) by nearest-neighbour assignment. Digital numbers were converted to reflectance as described above. The difference was defined as $\Delta\rho = \rho_\mathrm{ACOLITE} - \rho_\mathrm{Sen2Cor}$, so that positive values indicate higher ACOLITE reflectance.

Pixels were retained when the Sen2Cor digital number was positive, the ACOLITE value was valid, and the SCL class was not no-data, saturated or defective, cloud shadow, cloud, cirrus or snow/ice. Water, vegetation, bare-surface and other clear-sky classes were retained and no water-only mask was applied. Negative reflectance values were kept, because removing them would hide a feature of the products under comparison. For each band, the distribution of $\Delta\rho$ was summarised by its mean, median, standard deviation and percentiles, and by the proportion of pixels with $|\Delta\rho|$ above 0.01, 0.02, 0.05 and 0.10. To examine the spatial organisation of the differences, the most divergent 0.1% of pixels in each band (largest $|\Delta\rho|$) were assigned to mutually exclusive strata: the intertidal zone (a polygon layer delineating the intertidal flats), SCL water, SCL vegetation and SCL bare surfaces outside that polygon, and other pixels. The share of each stratum in the scene and the exceedance rate within each stratum were also computed, so that the composition of the extreme pixels could be read against the area covered by each stratum. Dependence of $\Delta\rho$ on surface brightness was examined by binning pixels according to Sen2Cor reflectance. As a check on resampling and edge effects, the analysis was repeated for the four native 10 m bands (B2, B3, B4 and B8) separately for SCL water and non-water pixels, after eroding the valid-pixel mask by one 10 m pixel.

Six 3 × 3-pixel windows (30 × 30 m) were selected to illustrate paired spectral signatures. They were chosen by fixed rules applied to the image: the window of minimum Sen2Cor near-infrared (B8) reflectance among SCL water pixels (T1); the window of maximum Sen2Cor red (B4) reflectance among SCL water pixels (T2); the windows of minimum B8 and of maximum B4 reflectance within the intertidal polygon (T3 and T4); the window of maximum Sen2Cor NDVI among SCL vegetation pixels (T5); and the window of maximum B4 reflectance among SCL bare-surface pixels (T6). The window means of ACOLITE and Sen2Cor reflectance were compared band by band. The targets are image-derived strata and carry no field-verified habitat labels. No reference measurement is available for this comparison, which therefore describes differences between the two processors and does not assess the accuracy of either. Calculations were performed in Python 3.12 with NumPy 1.26 and GDAL 3.8.

# 4. Results

## 4.1 Match-ups of atmospheric corrections

Both products reproduced the large reflectance contrast among surfaces (<a href="#fig-matchup" class="quarto-xref">Figure 3</a>). Over the 72 paired values, ACOLITE had an RMSD of 0.0500 (95% interval 0.0227–0.0698) and a mean bias of +0.0032 (−0.0258 to +0.0312), whereas Sen2Cor had an RMSD of 0.0659 (0.0402–0.0851) and a mean bias of +0.0309 (−0.0037 to +0.0635). The paired difference was $\Delta$RMSD = +0.0159 (+0.0017 to +0.0357); the intervals of the two processors overlapped broadly.

In [ ]:
knitr::include_graphics("figures/MatchupByBandAndSurface_Fig003.png")

The difference between processors was expressed mainly as a bias. Sen2Cor reflectance was higher than the field reflectance in all nine bands (mean bias +0.013 in B8A to +0.054 in B5; band RMSD 0.027–0.085), whereas ACOLITE biases were smaller and of mixed sign (−0.006 to +0.021; band RMSD 0.022–0.072) (<a href="#fig-residuals" class="quarto-xref">Figure 4</a>). ACOLITE had the lower RMSD at six of the eight stations (AC01, AC02, AC04, AC05, AC06 and AC08); Sen2Cor had the lower RMSD at AC07 (0.0126 versus 0.0219) and, marginally, at AC03 (0.0928 versus 0.0968). The contrast was concentrated at the brightest surfaces. Over white sand (AC06), Sen2Cor had an RMSD of 0.0950 and a bias of +0.0900, against 0.0229 and −0.0044 for ACOLITE; omitting AC06 reduced $\Delta$RMSD from +0.0159 to +0.0079, whereas omitting any other station gave +0.0152 to +0.0222. Both processors were positively biased over dry sand (Sen2Cor +0.072; ACOLITE +0.051). Over the two seagrass stations, RMSD was 0.0331 for ACOLITE and 0.0403 for Sen2Cor. At these stations the mean ACOLITE residual was positive in B1–B5 (+0.011 to +0.046) and negative in B6–B8A (−0.015 to −0.024), whereas the mean Sen2Cor residual was positive in B1–B5 (+0.014 to +0.054) and within ±0.009 in B6–B8A. Values for each surface class are given in <a href="#tbl-s2-surface" class="quarto-xref">Table 1</a>.

In [ ]:
knitr::include_graphics("figures/ResidualBoxplotsByBand_Fig004.png")

Spectral shape was preserved at most stations. The median spectral angle between the satellite and field spectra was 2.60° for ACOLITE and 2.83° for Sen2Cor (2.28° and 2.78° without AC03). Two stations departed from this pattern for both processors: AC03 (17.0° for ACOLITE; 20.5° for Sen2Cor) and AC08 (12.6°; 13.7°). At AC03, residuals in B1–B5 were between −0.050 and +0.002, but residuals in B6–B8A were between −0.12 and −0.16 for both processors, the two products returning similar reflectance (about 0.09–0.14) against a field reflectance of about 0.25. At AC08, satellite reflectance in B3 was 0.148–0.169 against a field value of 0.086, while B8A agreed closely (0.240–0.256 against 0.248).

## 4.2 Sensitivity of the match-up to analytical choices

Excluding AC03 lowered the RMSD of both processors (ACOLITE 0.0389, 0.0191–0.0578; Sen2Cor 0.0611, 0.0314–0.0826) and increased their mean biases (+0.0152 and +0.0449, respectively), while $\Delta$RMSD remained positive (+0.0222; +0.0075 to +0.0432). Extraction on a common 10 m grid left the result for ACOLITE unchanged (RMSD 0.0500) and gave an RMSD for Sen2Cor of 0.0659 with nearest-neighbour and 0.0680 with bilinear sampling. Using the mean of 3 × 3 pixels gave an RMSD of 0.0525 for ACOLITE and 0.0672 (nearest neighbour) and 0.0645 (bilinear) for Sen2Cor. The sign of $\Delta$RMSD was positive in all variants.

## 4.3 Pixel-wise comparison of Sen2Cor and ACOLITE reflectance

Approximately 7.31 million pixels per band were valid in both products. The scene-wide mean of $\Delta\rho$ was slightly negative in every band, from −0.0009 (B8A) to −0.0089 (B05), and its standard deviation ranged from 0.0072 (B1) to 0.0167 (B12). Absolute differences exceeded 0.01 in 14.8% (B8A) to 33.0% (B2) of pixels, exceeded 0.05 in 0.03% (B1) to 7.5% (B12) of pixels, and exceeded 0.10 in no more than 0.015% of pixels in any band. The small scene-wide means conceal marked spatial structure (<a href="#fig-pixel-maps" class="quarto-xref">Figure 5</a>). In all bands the differences were spatially coherent rather than random, and were organised along coastlines, tidal channels and the margins of the intertidal flats, where narrow bands of positive $\Delta\rho$ lay beside areas of negative $\Delta\rho$ in the visible and near-infrared bands. Negative differences of largest magnitude occurred over extensive areas in the south and east of the scene, most clearly in B5, B11 and B12.

The composition of the largest differences reflected this pattern. In B1–B7, 69.2–91.9% of the most divergent 0.1% of pixels were SCL bare surfaces outside the intertidal polygon and 5.7–27.4% lay within it, although bare surfaces covered 16.2% of valid pixels, the intertidal polygon 33.1% and SCL water outside the polygon 50.7%. The within-stratum exceedance rate was accordingly 0.43–0.57% for bare surfaces and 0.005–0.083% for the water and intertidal strata. In B12, the mean $\Delta\rho$ was −0.043 over bare surfaces and −0.002 over both the intertidal polygon and SCL water. The sign of the difference depended on surface brightness. In B4, the mean $\Delta\rho$ was +0.032 where Sen2Cor reflectance was negative, +0.012 where it was between 0 and 0.01, and −0.012 where it exceeded 0.4. The corresponding values in B8 were +0.019, +0.005 and −0.024. Sen2Cor produced negative reflectance in 0% (B11) to 0.86% (B6) of pixels, equivalent to 62,529 pixels in B6, whereas ACOLITE produced at most 72 negative pixels in any band. When the valid-pixel mask was eroded by one pixel, the mean $\Delta\rho$ of the four native 10 m bands remained negative for both SCL water (−0.0019 in B4 to −0.0069 in B2) and non-water pixels (−0.0060 in B3 and B4 to −0.0173 in B8).

The paired spectra of the six target windows are shown in <a href="#fig-pixel-spectra" class="quarto-xref">Figure 6</a>. Over the dark targets T1 (dark SCL water) and T3 (dark intertidal), ACOLITE reflectance exceeded Sen2Cor reflectance by 0.015–0.033 from B1 to B8A, with smaller differences in B11 (0.005–0.006) and B12 (0.002–0.003). Sen2Cor reflectance was negative at T1 in B4–B8A (down to −0.013 in B6) and at T3 in B6–B8A (down to −0.011 in B6), whereas ACOLITE reflectance was positive in all bands at both targets. Over the bright targets T2, T4 and T6, ACOLITE reflectance was lower than Sen2Cor reflectance in every band. The differences were largest in the red-edge and visible bands (T6, −0.154 in B5; T2, −0.120 in B2; T4, −0.086 in B5) and smallest in B1 for T4 (−0.008) and in B12 for T2 and T6 (−0.028 and −0.060). The shapes of the two spectra were similar at these targets, with the same increase from blue to near-infrared and the same decline in the shortwave infrared. At the vegetation target T5, both products showed the red-edge increase and near-infrared plateau, with ACOLITE reflectance higher by 0.019–0.028 in B1–B4 and by 0.001–0.010 in B5–B12 (0.002 in B8).

In [ ]:
knitr::include_graphics("figures/CorrectionDifferenceMaps_Fig005.png")

In [ ]:
knitr::include_graphics("figures/TargetSpectralSignatures_Fig006.png")

------------------------------------------------------------------------

# 5. Supplementary material

In [ ]:
s2_matchup <- readr::read_csv(
  "figures/sci24e_figure3_matchup_data.csv", show_col_types = FALSE
)
s2_surface <- s2_matchup |>
  dplyr::summarise(
    stations = dplyr::n_distinct(station),
    n = dplyr::n(),
    RMSD = sqrt(mean(residual^2)),
    Bias = mean(residual),
    MAE = mean(abs(residual)),
    .by = c(correction, target)
  ) |>
  dplyr::mutate(dplyr::across(c(RMSD, Bias, MAE), ~ sprintf("%.3f", .x))) |>
  dplyr::arrange(correction, target)
knitr::kable(s2_surface, align = c("l", "l", "r", "r", "r", "r", "r"))

Barbier, E.B., Hacker, S.D., Kennedy, C., Koch, E.W., Stier, A.C., Silliman, B.R., 2011. The value of estuarine and coastal ecosystem services. Ecological Monographs 81, 169–193. <https://doi.org/10.1890/10-1510.1>

Bui, Q.-T., Jamet, C., Vantrepotte, V., Mériaux, X., Cauvin, A., Mograne, M.A., 2022. Evaluation of Sentinel-2/MSI atmospheric correction algorithms over two contrasted french coastal waters. Remote Sensing 14, 1099. <https://doi.org/10.3390/rs14051099>

Davies, B.F.R., Gernez, P., Geraud, A., Oiry, S., Rosa, P., Zoffoli, M.L., Barillé, L., 2023. Multi- and hyperspectral classification of soft-bottom intertidal vegetation using a spectral library for coastal biodiversity remote sensing. Remote Sensing of Environment 290, 113554. <https://doi.org/10.1016/j.rse.2023.113554>

Davies, B.F.R., Oiry, S., Rosa, P., Zoffoli, M.L., Sousa, A.I., Thomas, O.R., Smale, D.A., Austen, M.C., Biermann, L., Attrill, M.J., Román, A., Navarro, G., Barillé, A.-L., Harin, N., Clewley, D., Martinez-Vicente, V., Gernez, P., Barillé, L., 2024a. A sentinel watching over inter-tidal seagrass phenology across western europe and north africa. Communications Earth & Environment 5, 382. <https://doi.org/10.1038/s43247-024-01543-z>

Davies, B.F.R., Oiry, S., Rosa, P., Zoffoli, M.L., Sousa, A.I., Thomas, O.R., Smale, D.A., Austen, M.C., Biermann, L., Attrill, M.J., Román, A., Navarro, G., Barillé, A.-L., Harin, N., Clewley, D., Martinez-Vicente, V., Gernez, P., Barillé, L., 2024b. Intertidal seagrass extent from sentinel-2 time-series show distinct trajectories in western europe. Remote Sensing of Environment 312, 114340. <https://doi.org/10.1016/j.rse.2024.114340>

<span class="nocase">de Fouw, J., Govers, L.L., van de Koppel, J., van Belzen, J., Dorigo, W., Sidi Cheikh, M.A., Christianen, M.J.A., van der Reijden, K.J., van der Geest, M., Piersma, T., Smolders, A.J.P., Olff, H., Lamers, L.P.M., van Gils, J.A., van der Heide, T.</span>, 2016. Drought, mutualism breakdown, and landscape-scale degradation of seagrass beds. Current Biology 26, 1051–1056. <https://doi.org/10.1016/j.cub.2016.02.023>

Doxani, G., Vermote, E.F., Roger, J.-C., Skakun, S., Gascon, F., Collison, A., De Keukelaere, L., Desjardins, C., Frantz, D., Hagolle, O., Kim, M., Louis, J., Pacifici, F., Pflug, B., Poilvé, H., Ramon, D., Richter, R., Yin, F., 2023. Atmospheric correction inter-comparison exercise, ACIX-II land: An assessment of atmospheric correction processors for Landsat 8 and Sentinel-2 over land. Remote Sensing of Environment 285, 113412. <https://doi.org/10.1016/j.rse.2022.113412>

Duffy, J.E., Benedetti-Cecchi, L., Trinanes, J., Muller-Karger, F.E., Ambo-Rappe, R., Boström, C., Buschmann, A.H., Byrnes, J., Coles, R.G., Creed, J., Cullen-Unsworth, L.C., Diaz-Pulido, G., Duarte, C.M., Edgar, G.J., Fortes, M., Goni, G., Hu, C., Huang, X., Hurd, C.L., Johnson, C., Konar, B., Krause-Jensen, D., Krumhansl, K., Macreadie, P., Marsh, H., McKenzie, L.J., Mieszkowska, N., Miloslavich, P., Montes, E., Nakaoka, M., Norderhaug, K.M., Nordlund, L.M., Orth, R.J., Prathep, A., Putman, N.F., Samper-Villarreal, J., Serrão, E.A., Short, F., Pinto, I.S., Steinberg, P., Stuart-Smith, R., Unsworth, R.K.F., Keulen, M. van, Tussenbroek, B.I. van, Wang, M., Waycott, M., Weatherdon, L.V., Wernberg, T., Yaakub, S.M., 2019. Toward a coordinated global observing system for seagrasses and marine macroalgae. Frontiers in Marine Science 6, 317. <https://doi.org/10.3389/fmars.2019.00317>

Dunic, J.C., Brown, C.J., Connolly, R.M., Turschwell, M.P., Côté, I.M., 2021. Long-term declines and recovery of meadow area across the world’s seagrass bioregions. Global Change Biology 27, 4096–4109. <https://doi.org/10.1111/gcb.15684>

El-Hacen, E.-H.M., Sidi Cheikh, M.A., Bouma, T.J., Olff, H., Piersma, T., 2020. Long-term changes in seagrass and benthos at banc d’arguin, mauritania, the premier intertidal system along the east atlantic flyway. Global Ecology and Conservation 24, e01364. <https://doi.org/10.1016/j.gecco.2020.e01364>

European Space Agency, 2021. [Sentinel-2 level-2A algorithm theoretical basis document](https://step.esa.int/thirdparties/sen2cor/2.10.0/docs/S2-PDGS-MPC-L2A-ATBD-V2.10.0.pdf) (No. S2-PDGS-MPC-L2A-ATBD). European Space Agency.

Field, C.A., Welsh, A.H., 2007. Bootstrapping clustered data. Journal of the Royal Statistical Society Series B: Statistical Methodology 69, 369–390. <https://doi.org/10.1111/j.1467-9868.2007.00593.x>

Fourqurean, J.W., Duarte, C.M., Kennedy, H., Marbà, N., Holmer, M., Mateo, M.A., Apostolaki, E.T., Kendrick, G.A., Krause-Jensen, D., McGlathery, K.J., Serrano, O., 2012. Seagrass ecosystems as a globally significant carbon stock. Nature Geoscience 5, 505–509. <https://doi.org/10.1038/ngeo1477>

Friese, C.A., Does, M. van der, Merkel, U., Iversen, M.H., Fischer, G., Stuut, J.-B.W., 2016. Environmental factors controlling the seasonal variability in particle size distribution of modern saharan dust deposited off cape blanc. Aeolian Research 22, 165–179. <https://doi.org/10.1016/j.aeolia.2016.04.005>

Friese, C.A., Hateren, J.A. van, Vogt, C., Fischer, G., Stuut, J.-B.W., 2017. Seasonal provenance changes in present-day saharan dust collected in and off mauritania. Atmospheric Chemistry and Physics 17, 10163–10193. <https://doi.org/10.5194/acp-17-10163-2017>

Kaufman, Y.J., Tanré, D., Boucher, O., 2002. A satellite view of aerosols in the climate system. Nature 419, 215–223. <https://doi.org/10.1038/nature01091>

Knippertz, P., Todd, M.C., 2012. Mineral dust aerosols over the sahara: Meteorological controls on emission and transport and implications for modeling. Reviews of Geophysics 50, RG1007. <https://doi.org/10.1029/2011RG000362>

Lyons, M.B., Roelfsema, C.M., Phinn, S.R., 2013. Towards understanding temporal and spatial dynamics of seagrass landscapes using time-series remote sensing. Estuarine, Coastal and Shelf Science 120, 42–53. <https://doi.org/10.1016/j.ecss.2013.01.015>

Main-Knorn, M., Pflug, B., Louis, J., Debaecker, V., Müller-Wilm, U., Gascon, F., 2017. Sen2Cor for Sentinel-2, in: Image and Signal Processing for Remote Sensing XXIII. SPIE, p. 1042704. <https://doi.org/10.1117/12.2278218>

Moulin, C., Gordon, H.R., Chomko, R.M., Banzon, V.F., Evans, R.H., 2001. Atmospheric correction of ocean color imagery through thick layers of saharan dust. Geophysical Research Letters 28, 5–8. <https://doi.org/10.1029/2000GL011803>

Nordlund, L.M., Koch, E.W., Barbier, E.B., Creed, J.C., 2016. Seagrass ecosystem services and their variability across genera and geographical regions. PLOS ONE 11, e0163091. <https://doi.org/10.1371/journal.pone.0163091>

Orth, R.J., Carruthers, T.J.B., Dennison, W.C., Duarte, C.M., Fourqurean, J.W., Heck, K.L., Hughes, A.R., Kendrick, G.A., Kenworthy, W.J., Olyarnik, S., Short, F.T., Waycott, M., Williams, S.L., 2006. A global crisis for seagrass ecosystems. BioScience 56, 987–996. [https://doi.org/10.1641/0006-3568(2006)56\[987:AGCFSE\]2.0.CO;2](https://doi.org/10.1641/0006-3568(2006)56[987:AGCFSE]2.0.CO;2)

Ould Dedah, S., 1993. Wind, surface water temperature, surface salinity and pollution in the area of the banc d’arguin, mauritania. Hydrobiologia 258, 9–19.

Pottier, A., Catry, T., Trégarot, E., Maréchal, J.-P., Fayad, V., David, G., Sidi Cheikh, M., Failler, P., 2021. Mapping coastal marine ecosystems of the national park of banc d’arguin (PNBA) in mauritania using Sentinel-2 imagery. International Journal of Applied Earth Observation and Geoinformation 102, 102419. <https://doi.org/10.1016/j.jag.2021.102419>

Roelfsema, C.M., Lyons, M., Kovacs, E.M., Maxwell, P., Saunders, M.I., Samper-Villarreal, J., Phinn, S.R., 2014. Multi-temporal mapping of seagrass cover, species and biomass: A semi-automated object based image analysis approach. Remote Sensing of Environment 150, 172–187. <https://doi.org/10.1016/j.rse.2014.05.001>

Senghor, H., Machu, É., Hourdin, F., Gaye, A.T., 2017. Seasonal cycle of desert aerosols in western africa: Analysis of the coastal transition with passive and active sensors. Atmospheric Chemistry and Physics 17, 8395–8410. <https://doi.org/10.5194/acp-17-8395-2017>

Song, C., Woodcock, C.E., Seto, K.C., Lenney, M.P., Macomber, S.A., 2001. Classification and change detection using landsat TM data: When and how to correct atmospheric effects? Remote Sensing of Environment 75, 230–244. <https://doi.org/10.1016/S0034-4257(00)00169-3>

Unsworth, R.K.F., McKenzie, L.J., Collier, C.J., Cullen-Unsworth, L.C., Duarte, C.M., Eklöf, J.S., Jarvis, J.C., Jones, B.L., Nordlund, L.M., 2019. Global challenges for seagrass conservation. Ambio 48, 801–815. <https://doi.org/10.1007/s13280-018-1115-y>

<span class="nocase">van der Heide, T., Govers, L.L., de Fouw, J., Olff, H., van der Geest, M., van Katwijk, M.M., Piersma, T., van de Koppel, J., Silliman, B.R., Smolders, A.J.P., van Gils, J.A.</span>, 2012. A three-stage symbiosis forms the foundation of seagrass ecosystems. Science 336, 1432–1434. <https://doi.org/10.1126/science.1219973>

Vanhellemont, Q., 2019. Adaptation of the dark spectrum fitting atmospheric correction for aquatic applications of the Landsat and Sentinel-2 archives. Remote Sensing of Environment 225, 175–192. <https://doi.org/10.1016/j.rse.2019.03.010>

Vanhellemont, Q., Ruddick, K., 2018. Atmospheric correction of metre-scale optical satellite data for inland and coastal water applications. Remote Sensing of Environment 216, 586–597. <https://doi.org/10.1016/j.rse.2018.07.015>

Warren, M.A., Simis, S.G.H., Martinez-Vicente, V., Poser, K., Bresciani, M., Alikas, K., Spyrakos, E., Giardino, C., Ansper, A., 2019. Assessment of atmospheric correction algorithms for the Sentinel-2A MultiSpectral Imager over coastal and inland waters. Remote Sensing of Environment 225, 267–289. <https://doi.org/10.1016/j.rse.2019.03.018>

Waycott, M., Duarte, C.M., Carruthers, T.J.B., Orth, R.J., Dennison, W.C., Olyarnik, S., Calladine, A., Fourqurean, J.W., Heck, K.L., Hughes, A.R., Kendrick, G.A., Kenworthy, W.J., Short, F.T., Williams, S.L., 2009. Accelerating loss of seagrasses across the globe threatens coastal ecosystems. Proceedings of the National Academy of Sciences 106, 12377–12381. <https://doi.org/10.1073/pnas.0905620106>

Wehrli, C., 1985. Extraterrestrial solar spectrum (Publication No. 615). Physikalisch-Meteorologisches Observatorium Davos; World Radiation Center, Davos Dorf, Switzerland.

Wolff, W.J., Van der Land, J., Nienhuis, P.H., De Wilde, P., 1993. The functioning of the ecosystem of banc d’arguin, mauritania: A review. Hydrobiologia 258, 211–222. <https://doi.org/10.1007/BF00006198>